# Complementary Error Analysis — Vision vs Text vs Ground Truth

The dual-expert claim rests on the two experts making **different** mistakes. This
notebook finds and characterizes those complementary cases on both datasets. For every
ground-truth cell it compares what the vision expert, the text expert, and the ground
truth say

In [ ]:
from pathlib import Path
import json, re
from collections import Counter
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import Levenshtein

DATASETS = {
    "kenny": {
        "gt_root":     Path("data/A25/input_images"),
        "vision_root": Path("data/outputs/vision_expert_gemma4_run"),
        "text_root":   Path("data/outputs/text_expert_gemma4_run"),
        "groups":      ["Biology", "CompSci", "ICDAR", "MatSci"],  
        "gt_subdir": "xmls", 
        "vision_subdir": "predictions", 
        "text_subdir": "nougat/predictions",
        "gt_format": "xml",
    },
    "SciTSR": {
        "gt_root":     Path("data/SciTSR/test"),
        "vision_root": Path("data/outputs/vision_expert_gemma4_SciTSR_run"),
        "text_root":   Path("data/outputs/text_expert_gemma4_SciTSR_run"),
        "groups":      ["test"],                                    
        "gt_subdir": "structure_processed", 
        "vision_subdir": "predictions", 
        "text_subdir": "predictions",
        "gt_format": "json",
    },
}
LEV_THRESHOLD = 1.0  
N_EXAMPLES    = 20     # examples shown per difference category

## Step 1 — Similar functionalities for calculating similarities

Identical to the router notebooks.

In [76]:
def normalize_text(text):
    if text is None:

        return ""
    
    text = str(text).lower()
    text = text.replace("\\times", "x")
    text = text.replace(chr(0x2212), "-").replace(chr(0x2013), "-").replace(chr(0x2014), "-")
    text = re.sub(r"\$\^\{(\d+)\}\$", r"\1", text)
    text = text.replace("$", "")
    return re.sub(r"\s+", " ", text).strip()

def lev_sim(a, b):

    a, b = normalize_text(a), normalize_text(b)

    if a == "" and b == "":
        return 1.0
    
    return 1.0 - Levenshtein.distance(a, b) / max(len(a), len(b), 1)

def cell_key(c):

    return (int(c["sr"]), int(c["er"]), int(c["sc"]), int(c["ec"]))

def parse_gt_xml(path):

    if path is None or not Path(path).exists():
        return []
    
    try:
        root = ET.parse(path).getroot()
    except Exception:
        return []
    
    out = []
    for c in root.findall("cell"):
        sr, sc = c.get("start_row"), c.get("start_col")

        if sr is None or sc is None:
            continue

        er, ec = c.get("end_row", sr), c.get("end_col", sc)
        t = c.find("text")

        out.append({
            "sr": int(sr), 
            "er": int(er), 
            "sc": int(sc), 
            "ec": int(ec),
            "text": normalize_text((t.text or "") if t is not None else "")
        })

    return out

def _scitsr_gt_text(cell):
    txt = cell.get("text")

    if not txt:
        content = cell.get("content")
        txt = " ".join(str(t) for t in content) if isinstance(content, list) else (str(content) if content else "")

    if not txt:
        txt = cell.get("tex", "")

    return normalize_text(txt)

def parse_gt_json(path):

    if path is None or not Path(path).exists():
        return []
    
    try:
        data = json.loads(Path(path).read_text(encoding="utf-8"))
    except Exception:
        return []
    
    cells = data.get("cells", data) if isinstance(data, dict) else data
    out = []
    for c in cells:
        sr, sc = c.get("start_row", c.get("sr")), c.get("start_col", c.get("sc"))

        if sr is None or sc is None:
            continue

        er, ec = c.get("end_row", c.get("er", sr)), c.get("end_col", c.get("ec", sc))
        out.append({
            "sr": int(sr), 
            "er": int(er), 
            "sc": int(sc), 
            "ec": int(ec),
            "text": _scitsr_gt_text(c)
        })

    return out

def parse_pred_json(path):
    if path is None or not Path(path).exists():
        return []
    
    try:
        data = json.loads(Path(path).read_text(encoding="utf-8"))
    except Exception:
        return []
    
    cells = data.get("cells", data) if isinstance(data, dict) else data
    if not isinstance(cells, list):
        return []
    
    out = []
    for c in cells:

        if not isinstance(c, dict):
            continue

        sr, sc = c.get("sr", c.get("start_row")), c.get("sc", c.get("start_col"))

        if sr is None or sc is None:
            continue

        er, ec = c.get("er", c.get("end_row", sr)), c.get("ec", c.get("end_col", sc))
        out.append({
            "sr": int(sr), 
            "er": int(er), 
            "sc": int(sc), 
            "ec": int(ec),
            "text": normalize_text(c.get("text", ""))
        })

    return out

def pred_lookup(cells_):
    out = {}

    for c in cells_:
        out.setdefault(cell_key(c), c)

    return out

def discover(ds):
    cfg, rows = DATASETS[ds], []
    for grp in cfg["groups"]:

        gt_dir  = cfg["gt_root"] / grp / cfg["gt_subdir"]
        vis_dir = cfg["vision_root"] / grp / cfg["vision_subdir"]
        txt_dir = cfg["text_root"] / grp / cfg["text_subdir"]
        ext = "*.xml" if cfg["gt_format"] == "xml" else "*.json"

        gt  = {p.stem: p for p in sorted(gt_dir.glob(ext))} if gt_dir.exists() else {}
        vis = {p.stem: p for p in vis_dir.glob("*.json")} if vis_dir.exists() else {}
        txt = {p.stem: p for p in txt_dir.glob("*.json")} if txt_dir.exists() else {}

        for s, g in gt.items():
            if s in vis and s in txt:
                rows.append({
                    "group": grp, 
                    "stem": s, 
                    "table_id": f"{grp}::{s}",
                    "gt_file": g, 
                    "vision_file": vis[s], 
                    "text_file": txt[s]
                })

    return pd.DataFrame(rows)

tables = {ds: discover(ds) for ds in DATASETS}
for ds, df in tables.items():
    print(f"{ds}: {len(df)} tables with both expert predictions")

kenny: 155 tables with both expert predictions
SciTSR: 157 tables with both expert predictions


## Step 2 — The symbolic/semantic classifier

In [77]:
# --- Symbolic vs semantic classification --------------------------------------
import unicodedata

LATEX_CMD_RE = re.compile(r"\\[a-zA-Z]+")

def _latex_cmd(m):
    """Resolve one \\command: \\pm -> ±, Greek-letter names -> the letter
    (unicode names them directly), anything else is dropped."""

    name = m.group(0)[1:]

    if name == "pm":
        return "±"
    
    if name == "lambda":
        name = "lamda"  # unicode's spelling

    try:
        return unicodedata.lookup(f"GREEK SMALL LETTER {name.upper()}")
    except KeyError:
        return " "

def semantic_skeleton(t):
    """Canonical content of a cell: symbols unified, markup stripped."""

    s = re.sub(r"(?<=[\d\s])pm(?=[\s\d])", "±", t)     
    s = LATEX_CMD_RE.sub(_latex_cmd, s)
    s = unicodedata.normalize("NFKC", s)                
    s = s.replace("−", "-")                        
    s = s.replace("×", "x").replace("·", " ").replace("∙", " ")
    s = re.sub(r"[{}\\$^_]", "", s)

    return re.sub(r"\s+", " ", s).strip()

def digits_of(s):
    return re.sub(r"\D", "", s)

def diff_category(a, b):
    """Classify how two normalized cell texts differ. Order matters."""
    if a == b:
        return "identical"

    if a == "" or b == "":
        return "empty content"

    sa, sb = semantic_skeleton(a), semantic_skeleton(b)
    if sa == sb or sa.replace(" ", "") == sb.replace(" ", ""):
        return "symbolic"

    if sa and sb and (sa in sb or sb in sa):
        return "truncation / merge"

   
    if (re.search(r"\d", sa) and re.search(r"\d", sb)
            and digits_of(sa) != digits_of(sb)
            and re.sub(r"[\d\s.,]", "", sa) == re.sub(r"[\d\s.,]", "", sb)):
        return "numeric value"

    return "other semantic"

def expert_vs_gt(expert_cell, gt_text):
    """How one expert's cell relates to the ground truth at that span."""
    if expert_cell is None:
        return "missing cell"

    cat = diff_category(normalize_text(expert_cell.get("text", "")), gt_text)

    return "correct" if cat == "identical" else cat


## Step 3 — One row per ground-truth cell, fully labeled

In [78]:

def build_analysis(ds):

    cfg = DATASETS[ds]
    gt_parse = parse_gt_xml if cfg["gt_format"] == "xml" else parse_gt_json

    rows = []
    for _, tab in tables[ds].iterrows():

        gt = gt_parse(tab["gt_file"])
        vl = pred_lookup(parse_pred_json(tab["vision_file"]))
        tl = pred_lookup(parse_pred_json(tab["text_file"]))

        for g in gt:

            k = cell_key(g)
            vc, tc = vl.get(k), tl.get(k)

            vt = "" if vc is None else vc["text"]
            tt = "" if tc is None else tc["text"]
            v_ok = vc is not None and lev_sim(g["text"], vt) >= LEV_THRESHOLD
            t_ok = tc is not None and lev_sim(g["text"], tt) >= LEV_THRESHOLD

            rows.append({
                "dataset": ds, 
                "group": tab["group"], 
                "table_id": tab["table_id"],
                "gt_text": g["text"], 
                "vision_text": vt, 
                "text_text": tt,
                "vision_present": vc is not None, 
                "text_present": tc is not None,
                "vision_correct": bool(v_ok), 
                "text_correct": bool(t_ok),
                "one_correct": bool(v_ok != t_ok),
                "winner": "vision" if (v_ok and not t_ok) else ("text" if (t_ok and not v_ok) else ("both" if v_ok else "neither")),
                "vt_diff": ("missing cell" if (vc is None) != (tc is None) else diff_category(vt, tt)),
                "vision_vs_gt": expert_vs_gt(vc, g["text"]),
                "text_vs_gt":   expert_vs_gt(tc, g["text"]),
            })
    return pd.DataFrame(rows)

df = pd.concat([build_analysis(ds) for ds in DATASETS], ignore_index=True)
print(df.groupby("dataset").agg(cells=("gt_text", "size"),
                                vision_acc=("vision_correct", "mean"),
                                text_acc=("text_correct", "mean"),
                                complementary=("one_correct", "mean")).round(4))

         cells  vision_acc  text_acc  complementary
dataset                                            
SciTSR    9287      0.8174    0.8637         0.1105
kenny     9491      0.7894    0.8035         0.1673


## Step 4 — The complementary examples themselves

Side-by-side GT / Vision / Text for every complementary cell (exactly one expert
correct), exported per dataset to `complementary_examples_<dataset>.csv`. Below, a few
examples per difference type.

In [79]:
# --- Export + show examples ------------------------------------------------------
cols = ["dataset", "group", "table_id", "gt_text", "vision_text", "text_text",
        "winner", "vt_diff", "vision_vs_gt", "text_vs_gt"]

for ds in DATASETS:

    comp = df[(df["dataset"] == ds) & (df["one_correct"])][cols]
    out = f"complementary_examples_{ds}.csv"
    comp.to_csv(out, index=False)

    print(f"{ds}: wrote {len(comp)} complementary cells -> {out}")

pd.set_option("display.max_colwidth", 40)
comp_all = df[df["one_correct"]]

for cat in comp_all["vt_diff"].value_counts().index:
    ex = comp_all[comp_all["vt_diff"] == cat]

    print(f"\n {cat} — {len(ex)} complementary cells")
    display(ex[["dataset", "gt_text", "vision_text", "text_text", "winner"]].head(N_EXAMPLES))

kenny: wrote 1588 complementary cells -> complementary_examples_kenny.csv
SciTSR: wrote 1026 complementary cells -> complementary_examples_SciTSR.csv

 symbolic — 700 complementary cells


,dataset,gt_text,vision_text,text_text,winner
4,kenny,niab elite magic1,niab elite magic1,niab elite magic ¹,vision
7,kenny,bmw magic2,bmw magic2,bmw magic ²,vision
10,kenny,apogee x paragon3,apogee x paragon3,apogee × paragon ³,vision
19,kenny,bmw magic2,bmw magic2,bmw magic ²,vision
22,kenny,niab elite magic1,niab elite magic1,niab elite magic ¹,vision
25,kenny,bmw magic2,bmw magic2,bmw magic ²,vision
31,kenny,paragon x cs5. reference genome8,paragon x cs5. reference genome8,paragon × cs ⁵. reference genome ⁸,vision
34,kenny,niab elite magic1. claire x malacca9...,niab elite magic1. claire x malacca9...,niab elite magic ¹. claire × malacca...,vision
36,kenny,dic12b †,dic12b†,dic12b †,text
40,kenny,bmw magic2,bmw magic2,bmw magic ²,vision



 numeric value — 687 complementary cells


,dataset,gt_text,vision_text,text_text,winner
86,kenny,0,0,2.8,vision
117,kenny,0.25 (0.07),0.25 (0.07),0.25(.07),vision
124,kenny,0.03 (0.02),0.03 (0.02),0.03(.02),vision
129,kenny,-0.05 (0.02),-0.05 (0.02),-0.05(.02),vision
132,kenny,0.24 (0.07),0.24 (0.07),0.24(.07),vision
134,kenny,0.09 (0.02),0.09 (0.02),0.09(.02),vision
137,kenny,0.24 (0.07),0.24 (0.07),0.24(.07),vision
139,kenny,-0.13 (0.02),-0.13 (0.02),-0.13(.02),vision
142,kenny,0.23 (0.06),0.23 (0.06),0.23(.06),vision
144,kenny,0.04 (0.02),0.04 (0.02),0.04(.02),vision



 missing cell — 679 complementary cells


...

,dataset,gt_text,vision_text,text_text,winner
45,kenny,fir13565,firl3565,fir13565,text
123,kenny,0.11 (0.17),0.11 (0.17),0.11(.017),vision
128,kenny,-0.31 (0.18),-0.31 (0.18),-0.31(.018),vision
133,kenny,0.23 (0.21),0.23 (0.21),0.23(.021),vision
138,kenny,-0.19 (0.19),-0.19 (0.19),-0.19(.019),vision
143,kenny,0.59 (0.20),0.59 (0.20),0.59(.020),vision
411,kenny,average marker distance (cm),average marker distance (cm),average markerª distance (cm),vision
497,kenny,number of variants with pre-existing...,number of variants with pre-existing...,number of existing personal and/or f...,vision
499,kenny,number of incidental variants (% if ...,number of incidental variants (% if ...,number of analysis criteria not iden...,vision
558,kenny,n/a,n/a,1 / 1,vision



 truncation / merge — 147 complementary cells


,dataset,gt_text,vision_text,text_text,winner
80,kenny,0,0,0.7,vision
83,kenny,0.7,0.7,0,vision
208,kenny,0.,0,0.,text
412,kenny,number of skewed markers,number of skewed markers,number of skewed,vision
498,kenny,number of variants with analysis cri...,number of variants with analysis cri...,number of,vision
1479,kenny,md loss,md,md loss,text
1495,kenny,rs-ped,ped,rs-ped,text
1511,kenny,rs-pce,pce,rs-pce,text
1527,kenny,rs-pjs,pjs,rs-pjs,text
1674,kenny,phased lstm,phased lstm,phased lstm clock-work rnn,vision



 empty content — 88 complementary cells


,dataset,gt_text,vision_text,text_text,winner
89,kenny,2.8,2.8,,vision
494,kenny,genes,genes,,vision
1178,kenny,,,98(89),vision
1240,kenny,1,1,,vision
1488,kenny,0.197,,0.197,text
1489,kenny,2.830,,2.830,text
1490,kenny,0.059,,0.059,text
1491,kenny,0.105,,0.105,text
1492,kenny,0.728,,0.728,text
1493,kenny,0.323,,0.323,text
